# P2-02 · Transformación de Consultas: HyDE + Query Decomposition
**Trabajo Práctico 2 — Ingeniería Ontológica 3010090**

Este notebook implementa dos técnicas avanzadas de transformación de consultas:

1. **HyDE** (Hypothetical Document Embeddings): Genera un documento hipotético que respondería a una pregunta corta/ambigua y lo usa para búsqueda semántica. Mejora el recall en consultas vagas.
2. **Query Decomposition**: Detecta preguntas complejas con múltiples condiciones y las descompone en subconsultas más simples que se resuelven de forma secuencial o en paralelo.

## Cuándo aplicar cada técnica
| Técnica | Tipo de consulta | Ejemplo |
|---|---|---|
| HyDE | Corta/ambigua (< 10 palabras) | `"BRCA1 mutations"` |
| Decomposition | Múltiple/condicional | `"What genes are mutated in breast cancer AND what drugs treat it?"` |
| Ninguna | Directa y clara | `"What is the mechanism of action of Remdesivir in COVID-19?"` |

In [ ]:
# ── Configuración local (reemplaza google.colab) ──────────────────────────
import sys
from pathlib import Path
# Ruta al repo resuelta desde este notebook (funciona para cualquier colaborador)
_REPO_ROOT = Path('__file__').parent.parent if '__file__' in dir() else Path().resolve()
# Buscar local_config.py subiendo directorios si hace falta
for _p in [_REPO_ROOT, _REPO_ROOT.parent, Path().resolve(), Path().resolve().parent]:
    if (_p / 'local_config.py').exists():
        _REPO_ROOT = _p
        break
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))
from local_config import (
    BASE_DIR, CORPUS_DIR, INDEX_DIR, TTL_PATH,
    GRAPHDB_BASE, REPO_NAME, GRAPHDB_URL,
    GOOGLE_API_KEY, GROQ_API_KEY, LANGCHAIN_API_KEY, TAVILY_API_KEY
)
print('✅ Configuración local cargada')
print(f'   BASE_DIR:    {BASE_DIR}')
print(f'   CORPUS_DIR:  {CORPUS_DIR}')
print(f'   GRAPHDB_URL: {GRAPHDB_URL}')


In [ ]:
# ── Modelos LLM ───────────────────────────────────────────────────────────
# Gemini 2.0 Flash: comprensión contextual profunda → clasifica y transforma consultas
gemini = ChatGoogleGenerativeAI(
    model='gemini-2.0-flash',
    temperature=0.2,
    max_tokens=2048
)

# Groq LLaMA 3.3 70B: velocidad alta → micro-decisiones y clasificación rápida
groq = ChatGroq(
    model='llama-3.3-70b-versatile',
    temperature=0.0,
    max_tokens=512
)

# Embeddings
embeddings = GoogleGenerativeAIEmbeddings(model='models/text-embedding-004')

# Cargar índice FAISS (creado en P2-01)
faiss_index = FAISS.load_local(
    str(INDEX_DIR / 'faiss_semantic'),
    embeddings,
    allow_dangerous_deserialization=True
)

print('✅ Modelos y FAISS cargados')


## 1 · Detector de tipo de consulta

Antes de aplicar HyDE o Decomposition, el sistema detecta automáticamente qué técnica es más apropiada.

In [ ]:
class QueryAnalysis(BaseModel):
    """Resultado del análisis de una consulta de usuario."""
    query_type: str = Field(description="Tipo: 'short_ambiguous', 'complex_multiple', 'direct'")
    needs_hyde: bool = Field(description="True si la consulta es corta/ambigua y se beneficia de HyDE")
    needs_decomposition: bool = Field(description="True si la consulta tiene múltiples preguntas o condiciones")
    reasoning: str = Field(description="Breve justificación de la clasificación")
    word_count: int = Field(description="Número de palabras en la consulta")


QUERY_ANALYSIS_PROMPT = ChatPromptTemplate.from_template("""
Eres un experto en análisis de consultas para sistemas RAG biomédicos.

Analiza la siguiente consulta y determina cómo debe transformarse:

Consulta: "{query}"

Criterios:
- short_ambiguous: consulta con menos de 10 palabras O que es vaga/ambigua sin contexto suficiente → aplicar HyDE
- complex_multiple: contiene múltiples preguntas (AND, OR, además, también) O preguntas condicionales → aplicar Decomposition  
- direct: pregunta clara y específica → sin transformación

Responde SOLO con el JSON solicitado.
""")

# Chain de análisis usando structured output de Gemini
query_analyzer = QUERY_ANALYSIS_PROMPT | gemini.with_structured_output(QueryAnalysis)


def analyze_query(query: str) -> QueryAnalysis:
    """Analiza una consulta y determina qué transformación aplicar."""
    result = query_analyzer.invoke({'query': query})
    return result


# Test
test_queries = [
    'BRCA1',                                                          # → HyDE
    'COVID-19 treatment',                                             # → HyDE
    'What genes are mutated in breast cancer and what drugs treat it?', # → Decomposition
    'Compare the efficacy of Remdesivir and Dexamethasone in COVID-19 patients AND analyze survival rates',  # → Decomposition
    'What is the mechanism of action of Trastuzumab in HER2-positive breast cancer?'  # → Direct
]

print('🔍 Análisis de consultas de prueba:\n')
for q in test_queries:
    analysis = analyze_query(q)
    print(f'Query: "{q}"')
    print(f'  Tipo: {analysis.query_type}')
    print(f'  HyDE: {analysis.needs_hyde} | Decomp: {analysis.needs_decomposition}')
    print(f'  Razón: {analysis.reasoning}')
    print()


## 2 · HyDE — Hypothetical Document Embeddings

### Algoritmo:
1. Recibe una consulta corta/ambigua
2. Un LLM genera un **documento hipotético** que respondería perfectamente esa consulta
3. Se calcula el embedding del documento hipotético (más rico semánticamente que la consulta corta)
4. Se usa ese embedding para buscar en el vector store

### Ventaja:
El documento hipotético contiene vocabulario y conceptos que no aparecen en la consulta original, mejorando el recall.

In [ ]:
HYDE_PROMPT = ChatPromptTemplate.from_template("""
Eres un experto en bioinformática y medicina. Escribe un párrafo técnico detallado (150-250 palabras) 
que sería la respuesta perfecta a la siguiente consulta biomédica.

Escribe el párrafo directamente, sin preámbulos ni introducciones. 
Usa terminología científica precisa.

Consulta: {query}

Párrafo hipotético:
""")

hyde_chain = HYDE_PROMPT | gemini | StrOutputParser()


def apply_hyde(query: str, vectorstore: FAISS, k: int = 5) -> dict:
    """
    Aplica HyDE a una consulta corta/ambigua.
    
    Args:
        query: Consulta original del usuario
        vectorstore: Índice FAISS con los chunks semánticos
        k: Número de documentos a recuperar
    
    Returns:
        dict con el documento hipotético y los resultados recuperados
    """
    # 1. Generar documento hipotético
    hypothetical_doc = hyde_chain.invoke({'query': query})
    
    # 2. Usar el embedding del doc hipotético para buscar
    results = vectorstore.similarity_search_with_score(hypothetical_doc, k=k)
    
    # 3. También buscar con la query original para comparación
    results_original = vectorstore.similarity_search_with_score(query, k=k)
    
    return {
        'original_query': query,
        'hypothetical_doc': hypothetical_doc,
        'hyde_results': results,
        'original_results': results_original
    }


# ── Ejemplo HyDE ──────────────────────────────────────────────────────────
short_query = 'BRCA1 breast cancer'
hyde_output = apply_hyde(short_query, faiss_index)

print(f'🧪 Query original: "{short_query}"\n')
print('📄 Documento hipotético generado:')
print('-' * 60)
print(hyde_output['hypothetical_doc'])
print('-' * 60)
print(f'\n📊 Comparación de resultados (Top-3):')
print('\n[Con HyDE]')
for i, (doc, score) in enumerate(hyde_output['hyde_results'][:3]):
    print(f'  {i+1}. {doc.metadata["doc_id"]} (score={score:.4f})')
print('\n[Sin HyDE - query original]')
for i, (doc, score) in enumerate(hyde_output['original_results'][:3]):
    print(f'  {i+1}. {doc.metadata["doc_id"]} (score={score:.4f})')


## 3 · Query Decomposition

### Algoritmo:
1. Recibe una consulta compleja con múltiples preguntas
2. Un LLM la descompone en subconsultas atómicas
3. Cada subconsulta se resuelve de forma independiente
4. Los resultados se combinan para generar una respuesta final

### Estrategias:
- **Secuencial**: cuando las subconsultas tienen dependencias
- **Paralelo**: cuando son independientes entre sí

In [ ]:
class SubQuery(BaseModel):
    """Una subconsulta atómica resultado de la descomposición."""
    sub_query: str = Field(description="La subconsulta atómica")
    depends_on: Optional[int] = Field(default=None, description="Índice de la subconsulta de la que depende (None=independiente)")
    reasoning: str = Field(description="Por qué se separó así")

class DecompositionResult(BaseModel):
    """Resultado completo de la descomposición de una consulta."""
    original_query: str = Field(description="Consulta original")
    sub_queries: List[SubQuery] = Field(description="Lista de subconsultas")
    execution_order: str = Field(description="'parallel' o 'sequential'")
    justification: str = Field(description="Explicación de la estrategia elegida")


DECOMP_PROMPT = ChatPromptTemplate.from_template("""
Eres un experto en sistemas RAG biomédicos. Tu tarea es descomponer consultas complejas 
en subconsultas atómicas que puedan responderse de forma independiente.

Consulta compleja: "{query}"

Instrucciones:
1. Identifica las preguntas individuales dentro de la consulta
2. Para cada subconsulta, determina si es independiente o depende de otra
3. Elige la estrategia de ejecución:
   - 'parallel': todas las subconsultas son independientes
   - 'sequential': alguna subconsulta depende del resultado de otra
4. Máximo 4 subconsultas

Responde SOLO con el JSON solicitado.
""")

decomp_chain = DECOMP_PROMPT | gemini.with_structured_output(DecompositionResult)


def decompose_query(query: str) -> DecompositionResult:
    """Descompone una consulta compleja en subconsultas atómicas."""
    return decomp_chain.invoke({'query': query})


def execute_decomposed_queries(decomp_result: DecompositionResult, 
                                vectorstore: FAISS, k: int = 3) -> dict:
    """
    Ejecuta las subconsultas y recopila los resultados.
    
    Soporta ejecución paralela y secuencial.
    """
    all_results = {}
    
    if decomp_result.execution_order == 'parallel':
        # Ejecutar todas las subconsultas en paralelo (aquí secuencialmente por simplicidad)
        for i, sq in enumerate(decomp_result.sub_queries):
            results = vectorstore.similarity_search_with_score(sq.sub_query, k=k)
            all_results[i] = {
                'sub_query': sq.sub_query,
                'results': results
            }
    else:
        # Ejecución secuencial: usar contexto previo si hay dependencia
        context_so_far = ''
        for i, sq in enumerate(decomp_result.sub_queries):
            enriched_query = sq.sub_query
            if sq.depends_on is not None and sq.depends_on in all_results:
                # Enriquecer la subconsulta con el contexto de la anterior
                prev_docs = all_results[sq.depends_on]['results'][:1]
                if prev_docs:
                    context_so_far = prev_docs[0][0].page_content[:500]
                    enriched_query = f'{context_so_far}\n\n{sq.sub_query}'
            
            results = vectorstore.similarity_search_with_score(enriched_query, k=k)
            all_results[i] = {
                'sub_query': sq.sub_query,
                'results': results
            }
    
    return all_results


# ── Ejemplo Decomposition ─────────────────────────────────────────────────
complex_query = 'What genes are mutated in breast cancer, what proteins do they encode, and what targeted therapies are available?'

decomp = decompose_query(complex_query)

print(f'🔬 Query compleja: "{complex_query}"\n')
print(f'📋 Estrategia: {decomp.execution_order}')
print(f'💡 Justificación: {decomp.justification}\n')
print('Subconsultas generadas:')
for i, sq in enumerate(decomp.sub_queries):
    dep = f' (depende de #{sq.depends_on})' if sq.depends_on is not None else ' (independiente)'
    print(f'  {i+1}. {sq.sub_query}{dep}')

# Ejecutar subconsultas
decomp_results = execute_decomposed_queries(decomp, faiss_index)

print('\n📊 Resultados por subconsulta:')
for idx, data in decomp_results.items():
    print(f'\n  [{idx+1}] "{data["sub_query"][:60]}..."')
    for doc, score in data['results'][:2]:
        print(f'       → {doc.metadata["doc_id"]} (score={score:.4f})')


## 4 · Router de transformación (función unificada)

In [ ]:
def transform_query(query: str, vectorstore: FAISS, k: int = 5) -> dict:
    """
    Router principal de transformación de consultas.
    
    Analiza la consulta y aplica automáticamente:
    - HyDE si es corta/ambigua
    - Query Decomposition si es compleja/múltiple
    - Sin transformación si es directa y clara
    
    Returns:
        dict con:
        - transform_type: qué transformación se aplicó
        - documents: lista de documentos recuperados
        - metadata: información de trazabilidad
    """
    # 1. Analizar la consulta
    analysis = analyze_query(query)
    
    metadata = {
        'original_query': query,
        'transform_type': 'none',
        'query_analysis': analysis.dict()
    }
    
    # 2. Aplicar transformación según el análisis
    if analysis.needs_hyde:
        # HyDE: consulta corta/ambigua
        hyde_result = apply_hyde(query, vectorstore, k=k)
        all_docs = [doc for doc, _ in hyde_result['hyde_results']]
        metadata['transform_type'] = 'hyde'
        metadata['hypothetical_doc'] = hyde_result['hypothetical_doc'][:300] + '...'
        
    elif analysis.needs_decomposition:
        # Query Decomposition: consulta compleja
        decomp = decompose_query(query)
        decomp_results = execute_decomposed_queries(decomp, vectorstore, k=3)
        
        # Combinar todos los documentos de las subconsultas (deduplicar)
        seen_ids = set()
        all_docs = []
        for data in decomp_results.values():
            for doc, _ in data['results']:
                doc_id = doc.metadata['chunk_id']
                if doc_id not in seen_ids:
                    seen_ids.add(doc_id)
                    all_docs.append(doc)
        
        metadata['transform_type'] = 'decomposition'
        metadata['sub_queries'] = [sq.sub_query for sq in decomp.sub_queries]
        metadata['execution_order'] = decomp.execution_order
        
    else:
        # Sin transformación: búsqueda directa
        results = vectorstore.similarity_search_with_score(query, k=k)
        all_docs = [doc for doc, _ in results]
        metadata['transform_type'] = 'direct'
    
    return {
        'documents': all_docs,
        'metadata': metadata
    }


# ── Demostración del router ───────────────────────────────────────────────
print('🎯 Demostración del Router de Transformación\n')

test_cases = [
    ('TP53',  'Esperado: HyDE'),
    ('What genes cause lung cancer AND how does Gefitinib inhibit EGFR AND what are the survival rates?', 'Esperado: Decomposition'),
    ('Describe the role of BRCA1 protein in DNA repair mechanism in breast cancer cells.', 'Esperado: Direct')
]

for query, expected in test_cases:
    print(f'Query: "{query[:70]}..."' if len(query) > 70 else f'Query: "{query}"')
    print(f'{expected}')
    result = transform_query(query, faiss_index, k=3)
    print(f'✅ Transformación aplicada: {result["metadata"]["transform_type"]}')
    print(f'   Documentos recuperados: {len(result["documents"])}')
    if 'hypothetical_doc' in result['metadata']:
        print(f'   Doc hipotético: {result["metadata"]["hypothetical_doc"][:100]}...')
    if 'sub_queries' in result['metadata']:
        print(f'   Subconsultas: {result["metadata"]["sub_queries"]}')
    print()


## Resumen

| Técnica | Cuándo | LLM usado | Beneficio |
|---|---|---|---|
| HyDE | Consulta < 10 palabras o ambigua | Gemini 2.0 Flash | +Recall en consultas vagas |
| Query Decomposition | Consulta con AND/OR o multi-pregunta | Gemini 2.0 Flash | Cubre todos los aspectos de la consulta |
| Sin transformación | Consulta directa y específica | — | Latencia mínima |

**Estas funciones serán integradas en el agente ReAct del notebook P2-04.**